# Offline staging for the air-gapped Blackwell box

Run this on a **connected** Linux session whose Python+torch **match the box**.
A free **Kaggle** session is ideal (it runs torch 2.10+cu128 on linux/cp312 — the
same as your RTX PRO 6000 box). It produces:

- `offline_bundle/wheels/` — every pip wheel the trainer needs (minus torch, which
  your box already has)
- `offline_bundle/repo/` — the pipeline code
- `offline_bundle/data/train_sft.jsonl` — the prebuilt SFT data (CPU-only)

Turn `offline_bundle/` into a **dataset** and attach it to the box. The ~63 GB model
is staged separately (last cell) — it's too big for this output.


## 0. Confirm this env matches the box
The `cp` tag and torch version here **must equal** the box's. If your box prints a different `python --version`, stage on a session that matches it.

In [ ]:
import sys, torch
print('python', sys.version.split()[0], '-> cp tag cp%d%d' % sys.version_info[:2])
print('torch ', torch.__version__, 'cuda', torch.version.cuda)
# On the BOX, run the same two lines and confirm they match before trusting the wheels.


## 1. Code

In [ ]:
%cd /kaggle/working
!rm -rf offline_bundle && mkdir -p offline_bundle
!rm -rf repo && git clone -q -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
!cp -r repo offline_bundle/repo
print('repo staged')


## 2. Wheel bundle (drop torch/CUDA — the box already has torch 2.10+cu128)

In [ ]:
import sys
W = '/kaggle/working/offline_bundle/wheels'
!mkdir -p {W}
# full closure for the training stack
!pip download -q -d {W} \
   'transformers>=4.45,<5' peft trl datasets accelerate einops sentencepiece \
   psutil safetensors huggingface_hub tokenizers hf_transfer
# remove the heavy wheels the box already provides (it has torch 2.10+cu128)
!rm -f {W}/torch-*.whl {W}/torchvision-*.whl {W}/torchaudio-*.whl {W}/triton-*.whl {W}/nvidia_*.whl
# mamba_ssm + causal_conv1d matching torch 2.10 / cu12 / abiTRUE (cp312 — change if your box differs)
CP = 'cp%d%d' % sys.version_info[:2]
cc = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu12torch2.10cxx11abiTRUE-{CP}-{CP}-linux_x86_64.whl'
mm = f'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-{CP}-{CP}-linux_x86_64.whl'
!wget -q -P {W} {cc} {mm} && echo 'mamba+causal wheels fetched for' {CP}
!ls -1 {W} | wc -l ; echo 'wheels total:'; du -sh {W}


## 3. Build the SFT data (pure pandas, no GPU)
Set your Kaggle KGAT token below (only needed here, on the connected machine).

In [ ]:
import os, glob, shutil, urllib.request
os.chdir('/kaggle/working/repo')
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if hits:
    shutil.copy(hits[0], 'data/train.csv'); print('train.csv (mount)')
else:
    TOK = 'KGAT_xxxxxxxxxxxxxxxx'   # <-- your token
    url='https://www.kaggle.com/api/v1/competitions/data/download/nvidia-nemotron-model-reasoning-challenge/train.csv'
    req=urllib.request.Request(url, headers={'Authorization': f'Bearer {TOK}'})
    open('data/train.csv','wb').write(urllib.request.urlopen(req).read())
    print('train.csv (download)')
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data
!mkdir -p /kaggle/working/offline_bundle/data && cp data/train_sft.jsonl /kaggle/working/offline_bundle/data/
import os; print('train_sft.jsonl rows:', sum(1 for _ in open('data/train_sft.jsonl')))


## 4. Finish the bundle
After this, **Save Version → Save & Run All**, then create a **Dataset** from the
output folder `offline_bundle/` (name it e.g. `nemotron-offline-bundle`). Attach that
dataset to the Blackwell box.


In [ ]:
!du -sh /kaggle/working/offline_bundle/* ; echo '---'; du -sh /kaggle/working/offline_bundle
print('Bundle ready at /kaggle/working/offline_bundle -> make it a dataset.')


## 5. The 63 GB model (stage separately)
`/kaggle/working` is too small for the model. Download it on a machine with ~70 GB
free (your Mac is fine — these are plain files), then ingest it as a dataset named
`nemotron-model` and attach it to the box:

```bash
pip install -U 'huggingface_hub[hf_transfer]'
HF_HUB_ENABLE_HF_TRANSFER=1 huggingface-cli download \
  nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16 \
  --local-dir ./nemotron-model --local-dir-use-symlinks False
```

The `./nemotron-model` folder (config.json, tokenizer, *.safetensors, the custom
`modeling_nemotron_h.py`) becomes the model dataset.
